# sceneid: embedder comparison on Colab

This notebook scores four embedders on **the same 6,230 Tier 1 test clips**, in one pass:

| Embedder | What it is |
|---|---|
| `clip-vit-b32` | OpenAI CLIP, the current default. Re-run here on the saved clips, which also checks that the published Tier 1 numbers reproduce |
| `sscd-disc-mixup` | Meta's SSCD, trained specifically for copy detection (including crops) |
| `dinov2-base` | Meta's DINOv2, self-supervised features that are strong at matching specific images |
| `phash64` | the classic perceptual hash: the no-machine-learning baseline |

**You need the Tier 1 run first**: this notebook reuses its answer key from `MyDrive/sceneid-bench`.

**Before you start:** *Runtime → Change runtime type* → a GPU (**T4** is enough). Rendering the clips is the
slow part and uses the CPU, so a machine with more CPU cores helps more than a faster GPU.

**No GPU available** ("Cannot connect to GPU backend")? Click *Connect without GPU* and run it anyway. Without a
GPU the notebook renders and saves every clip and scores `phash64`, which needs no GPU. Run it again when a GPU
is available: the clips are already saved, so only the three neural embedders remain, and that run is much shorter.

**Where things go** (all on your Drive, under `sceneid-bench/`)
- `clips/`: every rendered test clip, about 5 GB, saved the first time. **Later runs reuse them and skip rendering.**
- `compare/`: one library and one set of embeddings per embedder, plus `compare/results/`.

**If Colab disconnects:** run the cells from the top again. Finished clips, libraries and embeddings are all
on Drive, so it continues where it stopped.

**Time:** the first run renders every clip, so expect about as long as Tier 1 (3–4 hours on a 2-core
machine). Once the clips are saved, re-running or adding another embedder takes far less.

## 1. Check the GPU and connect Google Drive

In [ ]:
!nvidia-smi -L || echo 'No GPU in this session: see the note above'
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

import subprocess

HAS_GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
# Without a GPU, render and save every clip and score the hash baseline; the neural
# embedders run in a later session with a GPU, reusing the saved clips.
EMBEDDERS = "clip-vit-b32 sscd-disc-mixup dinov2-base phash64" if HAS_GPU else "phash64"
BRANCH = "main"
WORK = "/content/bench"  # films (Colab's own disk, rebuilt each session)
DRIVE = "/content/drive/MyDrive/sceneid-bench"  # from the Tier 1 run
STORE = f"{DRIVE}/compare"  # this run's libraries, embeddings and results
assert os.path.exists(f"{DRIVE}/queries.jsonl"), "Run the Tier 1 notebook first: it creates the answer key"

os.environ["SCENEID_BATCH_SIZE"] = "256"  # frames per GPU batch
os.environ["SCENEID_LOG_LEVEL"] = "INFO"

BENCH = (
    f"sceneid bench --workspace {WORK} --store {STORE} "
    f"--queries {DRIVE}/queries.jsonl --clips-dir {DRIVE}/clips"
)
# Clip rendering (ffmpeg) is the slow part and runs on the CPU, so use every core.
WORKERS = os.cpu_count() or 2
print(BENCH)
print(f"CPU cores: {WORKERS}")
print(f"GPU: {'yes' if HAS_GPU else 'no'} -> embedders this session: {EMBEDDERS}")

## 2. Install sceneid

In [ ]:
if not os.path.isdir("/content/sceneid"):
    !git clone --depth 1 -b {BRANCH} https://github.com/saarvesh0606/clip-to-video-scene-id.git /content/sceneid
else:
    !git -C /content/sceneid pull --ff-only
%cd /content/sceneid
!pip install -q -e ".[clip,bench]"
!ffmpeg -version | head -n 1
!git -C /content/sceneid log --oneline -1

## 3. Download the films (about 7 GB)

Colab's disk is empty in every new session, so the films download again (checksums verified).

In [ ]:
!{BENCH} download

## 4. Build one library per embedder

Each film is decoded once and its frames go to all four embedders. The SSCD and DINOv2 weights
download on first use (SSCD's file is checked against a pinned sha256).

In [ ]:
!{BENCH} index --embedders {EMBEDDERS}

## 5. Embed every query clip with every embedder

Run the trial cell first: it prints the speed and time remaining. Clips already saved in
`sceneid-bench/clips/` are reused; the rest are rendered once and saved there.
**If the session drops, reconnect and run everything from the top.**

In [ ]:
!{BENCH} embed --embedders {EMBEDDERS} --limit 40 --workers {WORKERS}

In [ ]:
!{BENCH} embed --embedders {EMBEDDERS} --workers {WORKERS}
!{BENCH} status --embedders {EMBEDDERS}

## 6. Score every embedder and compare them

Each embedder gets its own thresholds, tuned on the val split; everything reported comes from
the test split. This runs on the CPU and takes several minutes per embedder.

In [ ]:
RESULTS = f"{STORE}/results"
!{BENCH} evaluate --embedders {EMBEDDERS} --out-root {RESULTS}
dirs = " ".join(f"{RESULTS}/tier1-{e}" for e in EMBEDDERS.split())
!sceneid bench compare {dirs} --out {RESULTS}/comparison.md
from IPython.display import Markdown, display

display(Markdown(open(f"{RESULTS}/comparison.md", encoding="utf-8").read()))

## 7. Bring the results back

Run the cell below to download `compare-results.zip` (every embedder's report plus the
comparison), then send it over. It also stays on your Drive in `sceneid-bench/compare/results/`.
When you're done: *Runtime → Disconnect and delete runtime*.

In [ ]:
import shutil

from google.colab import files

zip_path = shutil.make_archive("/content/compare-results", "zip", RESULTS)
files.download(zip_path)